# Week 69, Day 3: Project: parser

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w069d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('tiny_ast.py', 'w', encoding='utf-8') as f:
    f.write('from dataclasses import dataclass\n\n@dataclass\nclass Number:\n    value: object\n    line: int = 0\n\n\n@dataclass\nclass String:\n    value: str\n    line: int = 0\n\n\n@dataclass\nclass Literal:\n    value: object\n    line: int = 0\n\n\n@dataclass\nclass Name:\n    name: str\n    line: int = 0\n\n\n@dataclass\nclass Unary:\n    op: str\n    operand: object\n    line: int = 0\n\n\n@dataclass\nclass Binary:\n    op: str\n    left: object\n    right: object\n    line: int = 0\n\n\n@dataclass\nclass Logical:\n    op: str\n    left: object\n    right: object\n    line: int = 0\n\n\n@dataclass\nclass Call:\n    callee: object\n    args: list\n    line: int = 0\n\n\n@dataclass\nclass Let:\n    name: str\n    value: object\n    line: int = 0\n\n\n@dataclass\nclass Assign:\n    name: str\n    value: object\n    line: int = 0\n\n\n@dataclass\nclass Print:\n    value: object\n    line: int = 0\n\n\n@dataclass\nclass ExprStmt:\n    value: object\n    line: int = 0\n\n\n@dataclass\nclass Block:\n    body: list\n    line: int = 0\n\n\n@dataclass\nclass If:\n    test: object\n    then: object\n    otherwise: object\n    line: int = 0\n\n\n@dataclass\nclass While:\n    test: object\n    body: object\n    line: int = 0\n\n\n@dataclass\nclass Function:\n    name: str\n    params: list\n    body: list\n    line: int = 0\n\n\n@dataclass\nclass Return:\n    value: object\n    line: int = 0\n')

### Exercise: Step 11: reading tokens

Start the `Parser(tokens)` class, which keeps the token list and a position `pos`. Write `peek()` (the current token), `at(kind, text=None)` (True if the current token has that kind and, if given, that text), `match(kind, text=None)` (if `at`, move past the token and return it, otherwise return `None`), and `expect(kind, text=None, what=None)`: like `match`, but if the token isn't there, raise `ParseError(f"expected {what or text or kind.lower()}, got {shown}")` at the current token's position, where `shown` is `"end of input"` for `EOF` and otherwise the token's text in `repr` form.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0


class Parser:
    def __init__(self, tokens):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_helpers():\n    \"peek, at, match and expect\"\n    p = Parser(tokenize(\"let x = 1;\"))\n    assert p.peek().kind == \"LET\" and p.at(\"LET\") and not p.at(\"NAME\") and p.pos == 0\n    assert p.match(\"NAME\") is None and p.pos == 0\n    assert p.match(\"LET\").text == \"let\" and p.pos == 1\n    assert p.expect(\"NAME\").text == \"x\" and p.at(\"OP\", \"=\") and not p.at(\"OP\", \";\")\n    assert p.match(\"OP\", \";\") is None and p.match(\"OP\", \"=\").text == \"=\"\n    try:\n        p.expect(\"OP\", \")\")\n        assert False\n    except ParseError as e:\n        assert str(e) == \"line 1, column 9: expected ), got '1'\"\n    p.match(\"NUMBER\"); p.match(\"OP\")\n    try:\n        p.expect(\"NAME\", what=\"a variable name\")\n        assert False\n    except ParseError as e:\n        assert e.message == \"expected a variable name, got end of input\"\n    try:\n        Parser(tokenize(\"5\")).expect(\"STRING\")\n        assert False\n    except ParseError as e:\n        assert e.message == \"expected string, got '5'\"")

### Exercise: Step 12: the tightest expressions

Recursive descent starts from the expressions that bind most tightly. Add three methods to the parser (whose helpers and a temporary `expression` are given): `primary()` reads a number, string, `true`/`false`/`nil`, name, or bracketed expression (`(` expression `)`), returning `Number`, `String`, `Literal` or `Name` nodes with the token's line, and otherwise raises `ParseError(f"expected an expression, got {shown}")` at the token; `call()` reads a primary followed by any number of argument lists, `f(a, b)(c)`, building `Call` nodes; and `unary()` reads any number of `-` or `not` in front of a call, building `Unary` nodes.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0


class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    def expression(self):
        return self.unary()      # Step 13 replaces this with the full chain

    def unary(self):
        ___

    def call(self):
        ___

    def primary(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import dataclasses\n\ndef _shape(node):\n    if isinstance(node, list):\n        return [_shape(n) for n in node]\n    if dataclasses.is_dataclass(node):\n        return (type(node).__name__,) + tuple(_shape(getattr(node, f.name)) for f in dataclasses.fields(node) if f.name != \"line\")\n    return node\n\ndef test_primary():\n    \"primary, call and unary\"\n    def expr(src):\n        return _shape(Parser(tokenize(src)).expression())\n    assert expr(\"42\") == (\"Number\", 42) and expr('\"hi\"') == (\"String\", \"hi\") and expr(\"nil\") == (\"Literal\", None)\n    assert expr(\"x\") == (\"Name\", \"x\") and expr(\"(((7)))\") == (\"Number\", 7)\n    assert expr(\"-x\") == (\"Unary\", \"-\", (\"Name\", \"x\")) and expr(\"not not true\") == (\"Unary\", \"not\", (\"Unary\", \"not\", (\"Literal\", True)))\n    assert expr(\"f()\") == (\"Call\", (\"Name\", \"f\"), [])\n    assert expr(\"f(1, y)(2)\") == (\"Call\", (\"Call\", (\"Name\", \"f\"), [(\"Number\", 1), (\"Name\", \"y\")]), [(\"Number\", 2)])\n    assert expr(\"-f(2)\") == (\"Unary\", \"-\", (\"Call\", (\"Name\", \"f\"), [(\"Number\", 2)]))\n    node = Parser(tokenize(\"\\n\\n  g(1)\")).expression()\n    assert node.line == 3 and node.callee.line == 3\n    for src, message in ((\")\", \"expected an expression, got ')'\"), (\"\", \"expected an expression, got end of input\"),\n                         (\"f(1\", \"expected ), got end of input\"), (\"(2\", \"expected ), got end of input\")):\n        try:\n            Parser(tokenize(src)).expression()\n            assert False, src\n        except ParseError as e:\n            assert e.message == message, e.message")

### Exercise: Step 13: precedence

Add the binary operators, loosest first: `expression` is `logic_or`; `logic_or` reads `logic_and`s separated by `or`, and `logic_and` reads `equality`s separated by `and`, both building `Logical` nodes; `equality` (`==`, `!=`), `comparison` (`<`, `<=`, `>`, `>=`), `term` (`+`, `-`) and `factor` (`*`, `/`, `%`) each read operands of the next tighter level separated by their operators, building `Binary` nodes that group to the **left** (`1 - 2 - 3` is `(1 - 2) - 3`). Write a shared helper `binary(operators, operand)` for the last four. The earlier methods are given.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0


class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    def unary(self):
        if self.at("OP", "-"):
            line = self.match("OP").line
            return Unary("-", self.unary(), line)
        if self.at("NOT"):
            line = self.match("NOT").line
            return Unary("not", self.unary(), line)
        return self.call()

    def call(self):
        node = self.primary()
        while self.at("OP", "("):
            line = self.match("OP").line
            args = []
            if not self.at("OP", ")"):
                args.append(self.expression())
                while self.match("OP", ","):
                    args.append(self.expression())
            self.expect("OP", ")")
            node = Call(node, args, line)
        return node

    def primary(self):
        tok = self.peek()
        if self.match("NUMBER"):
            return Number(tok.value, tok.line)
        if self.match("STRING"):
            return String(tok.value, tok.line)
        if tok.kind in ("TRUE", "FALSE", "NIL"):
            self.pos += 1
            return Literal(tok.value, tok.line)
        if self.match("NAME"):
            return Name(tok.text, tok.line)
        if self.match("OP", "("):
            node = self.expression()
            self.expect("OP", ")")
            return node
        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)
        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)

    def expression(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import dataclasses\n\ndef _shape(node):\n    if isinstance(node, list):\n        return [_shape(n) for n in node]\n    if dataclasses.is_dataclass(node):\n        return (type(node).__name__,) + tuple(_shape(getattr(node, f.name)) for f in dataclasses.fields(node) if f.name != \"line\")\n    return node\n\ndef test_precedence():\n    \"binary operators\"\n    def expr(src):\n        return _shape(Parser(tokenize(src)).expression())\n    one, two, three = (\"Number\", 1), (\"Number\", 2), (\"Number\", 3)\n    assert expr(\"1 + 2 * 3\") == (\"Binary\", \"+\", one, (\"Binary\", \"*\", two, three))\n    assert expr(\"1 - 2 - 3\") == (\"Binary\", \"-\", (\"Binary\", \"-\", one, two), three)\n    assert expr(\"(1 + 2) * 3\") == (\"Binary\", \"*\", (\"Binary\", \"+\", one, two), three)\n    assert expr(\"1 < 2 == true\") == (\"Binary\", \"==\", (\"Binary\", \"<\", one, two), (\"Literal\", True))\n    assert expr(\"a or b and c\") == (\"Logical\", \"or\", (\"Name\", \"a\"), (\"Logical\", \"and\", (\"Name\", \"b\"), (\"Name\", \"c\")))\n    assert expr(\"-1 * 2 % 3\") == (\"Binary\", \"%\", (\"Binary\", \"*\", (\"Unary\", \"-\", one), two), three)\n    assert expr(\"not a == b\") == (\"Binary\", \"==\", (\"Unary\", \"not\", (\"Name\", \"a\")), (\"Name\", \"b\"))\n    assert expr(\"f(1 + 2, 3 >= 4)\")[2][1] == (\"Binary\", \">=\", (\"Number\", 3), (\"Number\", 4))\n    try:\n        Parser(tokenize(\"1 + * 2\")).expression()\n        assert False\n    except ParseError as e:\n        assert e.message == \"expected an expression, got '*'\"")

### Exercise: Step 14: statements

Add statements to the parser (the expression methods are given): `statement()` reads `let name = expr;` (a `Let`), `print expr;` (a `Print`), a block `{ ... }` (a `Block` of the statements from `block()`), `name = expr;` (an `Assign`: a name followed by `=`, checked by looking at the token **after** the current one), or otherwise an expression followed by `;` (an `ExprStmt`). Every statement node gets the line of its first token, and every missing `;` is a `ParseError`. Then `block()`: `{`, statements until `}` (raising `ParseError("expected '}', got end of input")` if the input ends first), and `}`, returning the list; `program()`: statements until `EOF`; and a function `parse(source)` that tokenises and parses a whole program.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0


class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    def expression(self):
        return self.logic_or()

    def logic_or(self):
        node = self.logic_and()
        while self.at("OR"):
            line = self.match("OR").line
            node = Logical("or", node, self.logic_and(), line)
        return node

    def logic_and(self):
        node = self.equality()
        while self.at("AND"):
            line = self.match("AND").line
            node = Logical("and", node, self.equality(), line)
        return node

    def binary(self, operators, operand):
        node = operand()
        while self.peek().kind == "OP" and self.peek().text in operators:
            tok = self.match("OP")
            node = Binary(tok.text, node, operand(), tok.line)
        return node

    def equality(self):
        return self.binary({"==", "!="}, self.comparison)

    def comparison(self):
        return self.binary({"<", "<=", ">", ">="}, self.term)

    def term(self):
        return self.binary({"+", "-"}, self.factor)

    def factor(self):
        return self.binary({"*", "/", "%"}, self.unary)

    def unary(self):
        if self.at("OP", "-"):
            line = self.match("OP").line
            return Unary("-", self.unary(), line)
        if self.at("NOT"):
            line = self.match("NOT").line
            return Unary("not", self.unary(), line)
        return self.call()

    def call(self):
        node = self.primary()
        while self.at("OP", "("):
            line = self.match("OP").line
            args = []
            if not self.at("OP", ")"):
                args.append(self.expression())
                while self.match("OP", ","):
                    args.append(self.expression())
            self.expect("OP", ")")
            node = Call(node, args, line)
        return node

    def primary(self):
        tok = self.peek()
        if self.match("NUMBER"):
            return Number(tok.value, tok.line)
        if self.match("STRING"):
            return String(tok.value, tok.line)
        if tok.kind in ("TRUE", "FALSE", "NIL"):
            self.pos += 1
            return Literal(tok.value, tok.line)
        if self.match("NAME"):
            return Name(tok.text, tok.line)
        if self.match("OP", "("):
            node = self.expression()
            self.expect("OP", ")")
            return node
        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)
        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)

    def statement(self):
        ___

    def block(self):
        ___

    def program(self):
        ___


def parse(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import dataclasses\n\ndef _shape(node):\n    if isinstance(node, list):\n        return [_shape(n) for n in node]\n    if dataclasses.is_dataclass(node):\n        return (type(node).__name__,) + tuple(_shape(getattr(node, f.name)) for f in dataclasses.fields(node) if f.name != \"line\")\n    return node\n\ndef test_statements():\n    \"simple statements\"\n    prog = parse('let x = 1 + 2;\\nprint x;\\nx = x * 2;\\n{ let y = x; print y; }\\nf(x);')\n    assert _shape(prog) == [\n        (\"Let\", \"x\", (\"Binary\", \"+\", (\"Number\", 1), (\"Number\", 2))),\n        (\"Print\", (\"Name\", \"x\")),\n        (\"Assign\", \"x\", (\"Binary\", \"*\", (\"Name\", \"x\"), (\"Number\", 2))),\n        (\"Block\", [(\"Let\", \"y\", (\"Name\", \"x\")), (\"Print\", (\"Name\", \"y\"))]),\n        (\"ExprStmt\", (\"Call\", (\"Name\", \"f\"), [(\"Name\", \"x\")])),\n    ]\n    assert [s.line for s in prog] == [1, 2, 3, 4, 5] and parse(\"\") == [] and _shape(parse(\"{}\")) == [(\"Block\", [])]\n    for src, message in ((\"let = 1;\", \"expected a name, got '='\"), (\"print 1\", \"expected ;, got end of input\"),\n                         (\"{ print 1;\", \"expected '}', got end of input\"), (\"let x 1;\", \"expected =, got '1'\"),\n                         (\"x == 1\", \"expected ;, got end of input\")):\n        try:\n            parse(src)\n            assert False, src\n        except ParseError as e:\n            assert e.message == message, e.message")

### Exercise: Step 15: control flow and functions

Complete `statement()` (Step 14's version is given, with a gap) with four more kinds: `if (test) statement` with an optional `else statement` (an `If`, with `otherwise` `None` when there's no `else`); `while (test) statement` (a `While`); `fn name(a, b) { ... }` (a `Function` with the parameter names and the block's statements; error messages say `"a function name"` and `"a parameter name"`); and `return expr;` or `return;` (a `Return`, with value `None` for the bare form). An `else` belongs to the nearest `if`.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0


class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    def expression(self):
        return self.logic_or()

    def logic_or(self):
        node = self.logic_and()
        while self.at("OR"):
            line = self.match("OR").line
            node = Logical("or", node, self.logic_and(), line)
        return node

    def logic_and(self):
        node = self.equality()
        while self.at("AND"):
            line = self.match("AND").line
            node = Logical("and", node, self.equality(), line)
        return node

    def binary(self, operators, operand):
        node = operand()
        while self.peek().kind == "OP" and self.peek().text in operators:
            tok = self.match("OP")
            node = Binary(tok.text, node, operand(), tok.line)
        return node

    def equality(self):
        return self.binary({"==", "!="}, self.comparison)

    def comparison(self):
        return self.binary({"<", "<=", ">", ">="}, self.term)

    def term(self):
        return self.binary({"+", "-"}, self.factor)

    def factor(self):
        return self.binary({"*", "/", "%"}, self.unary)

    def unary(self):
        if self.at("OP", "-"):
            line = self.match("OP").line
            return Unary("-", self.unary(), line)
        if self.at("NOT"):
            line = self.match("NOT").line
            return Unary("not", self.unary(), line)
        return self.call()

    def call(self):
        node = self.primary()
        while self.at("OP", "("):
            line = self.match("OP").line
            args = []
            if not self.at("OP", ")"):
                args.append(self.expression())
                while self.match("OP", ","):
                    args.append(self.expression())
            self.expect("OP", ")")
            node = Call(node, args, line)
        return node

    def primary(self):
        tok = self.peek()
        if self.match("NUMBER"):
            return Number(tok.value, tok.line)
        if self.match("STRING"):
            return String(tok.value, tok.line)
        if tok.kind in ("TRUE", "FALSE", "NIL"):
            self.pos += 1
            return Literal(tok.value, tok.line)
        if self.match("NAME"):
            return Name(tok.text, tok.line)
        if self.match("OP", "("):
            node = self.expression()
            self.expect("OP", ")")
            return node
        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)
        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)

    def statement(self):
        tok = self.peek()
        if self.match("LET"):
            name = self.expect("NAME", what="a name").text
            self.expect("OP", "=")
            value = self.expression()
            self.expect("OP", ";")
            return Let(name, value, tok.line)
        if self.match("PRINT"):
            value = self.expression()
            self.expect("OP", ";")
            return Print(value, tok.line)
        if self.at("OP", "{"):
            return Block(self.block(), tok.line)
        # Step 15: if, while, fn and return go here
        ___
        if tok.kind == "NAME" and self.tokens[self.pos + 1].kind == "OP" and self.tokens[self.pos + 1].text == "=":
            self.pos += 2
            value = self.expression()
            self.expect("OP", ";")
            return Assign(tok.text, value, tok.line)
        value = self.expression()
        self.expect("OP", ";")
        return ExprStmt(value, tok.line)

    def block(self):
        self.expect("OP", "{")
        body = []
        while not self.at("OP", "}"):
            if self.at("EOF"):
                tok = self.peek()
                raise ParseError("expected '}', got end of input", tok.line, tok.column)
            body.append(self.statement())
        self.expect("OP", "}")
        return body

    def program(self):
        body = []
        while not self.at("EOF"):
            body.append(self.statement())
        return body


def parse(source):
    return Parser(tokenize(source)).program()

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import dataclasses\n\ndef _shape(node):\n    if isinstance(node, list):\n        return [_shape(n) for n in node]\n    if dataclasses.is_dataclass(node):\n        return (type(node).__name__,) + tuple(_shape(getattr(node, f.name)) for f in dataclasses.fields(node) if f.name != \"line\")\n    return node\n\ndef test_control_flow():\n    \"if, while, fn and return\"\n    prog = parse(\"\"\"\nfn fib(n) {\n    if (n < 2) return n;\n    return fib(n - 1) + fib(n - 2);\n}\nlet i = 0;\nwhile (i < 3) { print fib(i); i = i + 1; }\nif (i == 3) print \"done\"; else { print \"odd\"; }\nfn nothing() { return; }\n\"\"\")\n    fib, _, loop, choice, nothing = prog\n    assert _shape(fib)[:3] == (\"Function\", \"fib\", [\"n\"]) and fib.line == 2\n    assert _shape(fib.body[0]) == (\"If\", (\"Binary\", \"<\", (\"Name\", \"n\"), (\"Number\", 2)), (\"Return\", (\"Name\", \"n\")), None)\n    assert _shape(loop)[0] == \"While\" and _shape(loop.body)[0] == \"Block\" and len(loop.body.body) == 2\n    assert _shape(choice)[2] == (\"Print\", (\"String\", \"done\")) and _shape(choice)[3] == (\"Block\", [(\"Print\", (\"String\", \"odd\"))])\n    assert _shape(nothing) == (\"Function\", \"nothing\", [], [(\"Return\", None)])\n    nested = parse(\"if (a) if (b) print 1; else print 2;\")[0]\n    assert nested.otherwise is None and nested.then.otherwise is not None\n    assert _shape(parse(\"fn f(a, b, c) {}\")[0]) == (\"Function\", \"f\", [\"a\", \"b\", \"c\"], [])\n    for src, message in ((\"fn (x) {}\", \"expected a function name, got '('\"), (\"fn f(1) {}\", \"expected a parameter name, got '1'\"),\n                         (\"while x print 1;\", \"expected (, got 'x'\"), (\"fn f() print 1;\", \"expected {, got 'print'\")):\n        try:\n            parse(src)\n            assert False, src\n        except ParseError as e:\n            assert e.message == message, e.message")

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0


class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    # --- expressions, lowest precedence first ---
    def expression(self):
        return self.logic_or()

    def logic_or(self):
        node = self.logic_and()
        while self.at("OR"):
            line = self.match("OR").line
            node = Logical("or", node, self.logic_and(), line)
        return node

    def logic_and(self):
        node = self.equality()
        while self.at("AND"):
            line = self.match("AND").line
            node = Logical("and", node, self.equality(), line)
        return node

    def binary(self, operators, operand):
        node = operand()
        while self.peek().kind == "OP" and self.peek().text in operators:
            tok = self.match("OP")
            node = Binary(tok.text, node, operand(), tok.line)
        return node

    def equality(self):
        return self.binary({"==", "!="}, self.comparison)

    def comparison(self):
        return self.binary({"<", "<=", ">", ">="}, self.term)

    def term(self):
        return self.binary({"+", "-"}, self.factor)

    def factor(self):
        return self.binary({"*", "/", "%"}, self.unary)

    def unary(self):
        if self.at("OP", "-"):
            line = self.match("OP").line
            return Unary("-", self.unary(), line)
        if self.at("NOT"):
            line = self.match("NOT").line
            return Unary("not", self.unary(), line)
        return self.call()

    def call(self):
        node = self.primary()
        while self.at("OP", "("):
            line = self.match("OP").line
            args = []
            if not self.at("OP", ")"):
                args.append(self.expression())
                while self.match("OP", ","):
                    args.append(self.expression())
            self.expect("OP", ")")
            node = Call(node, args, line)
        return node

    def primary(self):
        tok = self.peek()
        if self.match("NUMBER"):
            return Number(tok.value, tok.line)
        if self.match("STRING"):
            return String(tok.value, tok.line)
        if tok.kind in ("TRUE", "FALSE", "NIL"):
            self.pos += 1
            return Literal(tok.value, tok.line)
        if self.match("NAME"):
            return Name(tok.text, tok.line)
        if self.match("OP", "("):
            node = self.expression()
            self.expect("OP", ")")
            return node
        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)
        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)

    # --- statements ---
    def statement(self):
        tok = self.peek()
        if self.match("LET"):
            name = self.expect("NAME", what="a name").text
            self.expect("OP", "=")
            value = self.expression()
            self.expect("OP", ";")
            return Let(name, value, tok.line)
        if self.match("PRINT"):
            value = self.expression()
            self.expect("OP", ";")
            return Print(value, tok.line)
        if self.at("OP", "{"):
            return Block(self.block(), tok.line)
        if self.match("IF"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            then = self.statement()
            otherwise = self.statement() if self.match("ELSE") else None
            return If(test, then, otherwise, tok.line)
        if self.match("WHILE"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            return While(test, self.statement(), tok.line)
        if self.match("FN"):
            name = self.expect("NAME", what="a function name").text
            self.expect("OP", "(")
            params = []
            if not self.at("OP", ")"):
                params.append(self.expect("NAME", what="a parameter name").text)
                while self.match("OP", ","):
                    params.append(self.expect("NAME", what="a parameter name").text)
            self.expect("OP", ")")
            return Function(name, params, self.block(), tok.line)
        if self.match("RETURN"):
            value = None if self.at("OP", ";") else self.expression()
            self.expect("OP", ";")
            return Return(value, tok.line)
        if tok.kind == "NAME" and self.tokens[self.pos + 1].kind == "OP" and self.tokens[self.pos + 1].text == "=":
            self.pos += 2
            value = self.expression()
            self.expect("OP", ";")
            return Assign(tok.text, value, tok.line)
        value = self.expression()
        self.expect("OP", ";")
        return ExprStmt(value, tok.line)

    def block(self):
        self.expect("OP", "{")
        body = []
        while not self.at("OP", "}"):
            if self.at("EOF"):
                tok = self.peek()
                raise ParseError("expected '}', got end of input", tok.line, tok.column)
            body.append(self.statement())
        self.expect("OP", "}")
        return body

    def program(self):
        body = []
        while not self.at("EOF"):
            body.append(self.statement())
        return body


def parse(source):
    return Parser(tokenize(source)).program()

import dataclasses

def show_tree(node, depth=0):
    pad = "  " * depth
    if isinstance(node, list):
        for item in node:
            show_tree(item, depth)
        return
    fields = [(f.name, getattr(node, f.name)) for f in dataclasses.fields(node) if f.name != "line"]
    simple = [f"{k}={v!r}" for k, v in fields if not dataclasses.is_dataclass(v) and not isinstance(v, list) and v is not None]
    print(pad + type(node).__name__ + (" " + " ".join(simple) if simple else ""))
    for k, v in fields:
        if dataclasses.is_dataclass(v) or (isinstance(v, list) and v and dataclasses.is_dataclass(v[0])):
            show_tree(v, depth + 1)

show_tree(parse("let total = 1 + 2 * 3;\nif (total > 5) print \"big\"; else print total;"))

> **Why recursive descent?.** Parser generators can build a parser from a grammar automatically (CPython has used one, PEG-based, since 3.9). But hand-written recursive descent is still everywhere, because it's just ordinary code: you can read it, debug it with print statements, and give it error messages as precise as you like ("expected a parameter name, got '1'"). The cost is that the grammar is spread through the methods instead of written in one place, which is why it's worth keeping the grammar as a comment at the top of the parser.

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

The tokeniser is finished. From today, its code is in `tiny_lex.py`, and each exercise gives you everything it needs.

## The grammar and the tree

The **grammar** says how tokens may be combined. Written in a common notation (`*` means "any number of", `?` means "optional", `|` means "or"), Tiny's is:

```text
program     := statement* EOF
statement   := "let" NAME "=" expression ";"
             | "print" expression ";"
             | "if" "(" expression ")" statement ("else" statement)?
             | "while" "(" expression ")" statement
             | "fn" NAME "(" (NAME ("," NAME)*)? ")" block
             | "return" expression? ";"
             | NAME "=" expression ";"
             | block
             | expression ";"
block       := "{" statement* "}"
expression  := logic_or
logic_or    := logic_and ("or" logic_and)*
logic_and   := equality ("and" equality)*
equality    := comparison (("==" | "!=") comparison)*
comparison  := term (("<" | "<=" | ">" | ">=") term)*
term        := factor (("+" | "-") factor)*
factor      := unary (("*" | "/" | "%") unary)*
unary       := ("-" | "not") unary | call
call        := primary ("(" (expression ("," expression)*)? ")")*
primary     := NUMBER | STRING | "true" | "false" | "nil" | NAME | "(" expression ")"
```

Each rule becomes one method, and a rule that mentions another calls its method. **Precedence** comes from the order: `term` calls `factor` for its operands, so `*` groups before `+`. **Grouping to the left** comes from the loops: `term` reads `1`, then `- 2` to make `1 - 2`, then `- 3` to make `(1 - 2) - 3`.

The **syntax tree** is made of dataclasses, one per kind of node, each with a `line` for runtime error messages. They're given (they're just data):

## Step 11: reading tokens

Four small helpers do all the token handling, so the grammar methods can read like the grammar.

## Step 12: the tightest expressions

## Step 13: precedence

## Step 14: statements

## Step 15: control flow and functions

## Checkpoint

The finished parser, printing a tree with indentation:

## Quiz

## Before you go

Add a rule to the grammar on paper for a `**` power operator that binds more tightly than `*` and groups to the **right** (`2 ** 3 ** 2` is `2 ** (3 ** 2)`): which method would call which, and how does right grouping differ from the loop? Tomorrow: the evaluator, which runs the trees.